# Movie Agent — Complete Agentic Loop (#2.4~2.7)

OpenAI `tools` 파라미터 + `tool_calls` 로 만드는 **완전한 에이전트 루프**.

- API base: `https://nomad-movies-2.nomadcoders.workers.dev`
- 모델: `gpt-4o-mini`
- 도구 3개: `get_popular_movies()` · `get_movie_details(id)` · `get_similar_movies(id)`

**조건 5개:** `tools` 파라미터 / `tool_calls` 처리 / 실제 API 호출 후 결과 재전달 / 메모리 멀티턴 / **최종 답변까지 루프**

> 학습법: 각 코드셀은 채워져 있다. **🔮 실행 전 예측**을 머릿속(or 주석)에 적고 → 돌려서 맞춰보고 → **🔧 한 군데 바꿔** 또 예측. 연습_2.4-2.7 의 확장판이다 (1회 왕복 → while 루프).

## 0. 준비 — import & 클라이언트

In [26]:
import os, json, requests, openai
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=key)
print("key loaded:", bool(key))

key loaded: True


## 1. 실제 API 함수 3개 + 디스패처

movie-agent 에서 가져오되 **credits → `get_similar_movies` 로 교체**(`/similar`). 모델은 이 함수를 직접 못 돌린다 — 이름만 고르고 실행은 내 코드가.

🔮 **예측:** `FUNCTION_MAP["get_similar_movies"]` (괄호 X) 와 `FUNCTION_MAP["get_similar_movies"](693134)` — 각각 뭐가 나올까?

In [27]:
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(movie_id):
    return requests.get(f"{BASE_URL}/movies/{movie_id}").json()

def get_similar_movies(movie_id):                       # 🔧 credits 대신 similar
    return requests.get(f"{BASE_URL}/movies/{movie_id}/similar").json()

FUNCTION_MAP = {                                        # 이름(str) → 함수객체(괄호 X!)
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}
print("함수 준비:", list(FUNCTION_MAP))

함수 준비: ['get_popular_movies', 'get_movie_details', 'get_similar_movies']


## 2. `tools` 스키마 등록 (#2.4 — 수동 프롬프팅 폐기)

구식(movie-agent)은 SYSTEM_PROMPT 로 "JSON 으로만 답해" 강제했다. 이제는 **정식 스키마**로 등록 → OpenAI 가 알아서 모델에 주입. 각 함수의 `description` 이 모델의 **유일한 선택 근거**.

🔮 **예측:** `get_similar_movies` 의 description 을 `""` 로 비우면 "비슷한 영화" 질문에 모델이 그래도 이 도구를 고를까?

🔧 **수정 미션:** 셋 중 하나 description 을 일부러 모호하게(`"영화"`) 바꿔 돌려보고, 엉뚱한 도구를 고르는지 확인.

In [28]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "지금 인기 있는 영화 목록을 가져온다. 인자 없음.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 한 편의 상세 정보(줄거리·장르·예산 등)를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {"movie_id": {"type": "integer", "description": "영화의 id"}},
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화와 비슷한/추천 영화 목록을 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {"movie_id": {"type": "integer", "description": "기준이 되는 영화의 id"}},
                "required": ["movie_id"],
            },
        },
    },
]
print("등록된 도구:", [t["function"]["name"] for t in TOOLS])

등록된 도구: ['get_popular_movies', 'get_movie_details', 'get_similar_movies']


## 3. 에이전트 루프 (핵심!) — `while` 로 최종 답까지

연습_2.4-2.7 은 `1차 호출 → 실행 → 2차 호출` 로 **한 번만** 왕복했다. 여기선 모델이 도구를 **연쇄로** 부를 수 있으니 (예: id 모르면 popular 먼저 → 그 결과로 details) **루프**를 돈다.

**규칙:** 응답에 `tool_calls` 가 있으면 → 실행하고 결과 넣고 **한 바퀴 더**. `tool_calls` 가 없으면(=`content` 에 자연어) → 그게 **최종 답**, 루프 탈출.

🔮 **예측:** "인기 영화 알려줘" 한 번에 while 이 **몇 바퀴** 돌까? 도구 필요 없는 "고마워" 는?

🔧 **수정 미션:** `while True` 를 `for _ in range(1)` 로 바꾸면(=1바퀴만) 어떤 질문에서 최종 답이 안 나오고 끊길까?

In [29]:
SYSTEM_PROMPT = """너는 영화 전문가 에이전트다.
- 사용자 질문에 답하려면 등록된 도구로 실제 데이터를 가져와라.
- 영화를 나열/언급할 때는 반드시 제목과 id 를 "제목 (ID: 123)" 형식으로 함께 보여줘라. (사용자가 그 id 로 후속 질문을 하게)
- id 를 모르면 먼저 인기 목록/상세를 조회해 알아낸 뒤 다음 도구를 불러라.
- 도구가 준 데이터만 근거로, 한국어로 자연스럽게 답한다. 지어내지 마라."""

messages = [{"role": "system", "content": SYSTEM_PROMPT}]   # ← 전역 메모리 (멀티턴)

def agent(user_input):
    messages.append({"role": "user", "content": user_input})
    print(f"🙋 {user_input}")
    while True:
        res = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages, tools=TOOLS,
        )
        msg = res.choices[0].message
        messages.append(msg)                              # AI 응답(요청이든 답이든) 메모리에

        if not msg.tool_calls:                            # tool_calls 없음 = 최종 답
            print(f"🤖 {msg.content}\n")
            return msg.content

        for tc in msg.tool_calls:                         # 있으면 도구 실행 후 또 한 바퀴
            name = tc.function.name
            args = json.loads(tc.function.arguments)      # JSON 문자열 → dict
            print(f"  🔧 {name}({args})")
            result = FUNCTION_MAP[name](**args)           # **args = 키워드 인자로 펼침
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,                     # 어느 요청에 대한 결과인지 매칭
                "content": json.dumps(result, ensure_ascii=False)[:4000],
            })

print("agent 준비 완료")

agent 준비 완료


## 4. 멀티턴 시나리오 (제출용 — 출력 찍힌 상태로 커밋)

메모리가 살아있으면 2·3턴의 "첫 번째"·"그거" 가 **앞 턴 도구 결과(영화 id)** 를 가리킬 수 있다.

🔮 **예측:** 2턴 "첫 번째 영화 자세히" 에서 모델이 id 를 어디서 알아낼까? (1턴 popular 결과가 messages 에 있다)

In [ ]:
agent("지금 인기 있는 영화 알려줘")

🙋 지금 인기 있는 영화 알려줘
  🔧 get_popular_movies({})


In [ ]:
agent("그 중 첫 번째 영화 자세히 알려줘")

🙋 그 중 첫 번째 영화 자세히 알려줘
  🔧 get_movie_details({'movie_id': 1339713})
🤖 영화 **Obsession** (ID: 1339713)의 자세한 정보는 다음과 같습니다:

- **개요**: 신비로운 "One Wish Willow"를 깨고 사랑하는 사람을 얻으려는 낭만적인 남자가 그의 소원이 이루어지는 것을 경험하지만, 어떤 욕망이 어두운 대가를 치르게 할 수 있다는 것을 알게 됩니다.
- **장르**: 호러 (Horror), 스릴러 (Thriller)
- **런타임**: 108분
- **예산**: $750,000
- **수익**: $242,663,965
- **출시일**: 2026-05-13
- **평점**: 7.892 (투표 수: 724)
- **태그라인**: "조심해, 네가 원하는 것을 조심해..."
- **제작사**:
  - Tea Shop Productions (GB)
  - Under the Shell (US)
  - Capstone Pictures (US)
  - Blumhouse Productions (US)
- **제작국가**: 
  - 미국 (US)
  - 영국 (GB)
- **원어**: 영어
- **웹사이트**: [공식 홈페이지](http://www.focusfeatures.com/obsession)
  
![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

이 영화에 대해 더 궁금한 부분이 있거나 다른 질문이 있으시면 말씀해 주세요!



'영화 **Obsession** (ID: 1339713)의 자세한 정보는 다음과 같습니다:\n\n- **개요**: 신비로운 "One Wish Willow"를 깨고 사랑하는 사람을 얻으려는 낭만적인 남자가 그의 소원이 이루어지는 것을 경험하지만, 어떤 욕망이 어두운 대가를 치르게 할 수 있다는 것을 알게 됩니다.\n- **장르**: 호러 (Horror), 스릴러 (Thriller)\n- **런타임**: 108분\n- **예산**: $750,000\n- **수익**: $242,663,965\n- **출시일**: 2026-05-13\n- **평점**: 7.892 (투표 수: 724)\n- **태그라인**: "조심해, 네가 원하는 것을 조심해..."\n- **제작사**:\n  - Tea Shop Productions (GB)\n  - Under the Shell (US)\n  - Capstone Pictures (US)\n  - Blumhouse Productions (US)\n- **제작국가**: \n  - 미국 (US)\n  - 영국 (GB)\n- **원어**: 영어\n- **웹사이트**: [공식 홈페이지](http://www.focusfeatures.com/obsession)\n  \n![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)\n\n이 영화에 대해 더 궁금한 부분이 있거나 다른 질문이 있으시면 말씀해 주세요!'

In [ ]:
agent("그 영화랑 비슷한 영화 추천해줘")

🙋 그 영화랑 비슷한 영화 추천해줘
  🔧 get_similar_movies({'movie_id': 1339713})
🤖 영화 **Obsession**와 비슷한 추천 영화 목록은 다음과 같습니다:

1. **Wolf House** (ID: 433854)
   - 개요: 여섯 명의 친구들이 캠핑 중 사스콧을 발견하고 죽였다고 생각하지만, 실제로 그들이 풀어놓은 것은 더 악랄하고 고대의 초자연적인 공포입니다.
   - 출시일: 2017-01-03
   - 평점: 3.4
   ![Wolf House](https://image.tmdb.org/t/p/w780/cSa5LoVxtHtKOzuOIWcNqZsXkzz.jpg)

2. **Maria Full of Grace** (ID: 436)
   - 개요: 임신한 콜롬비아 10대가 가족을 위해 절실히 필요한 돈을 벌기 위해 마약 밀매원이 되는 이야기입니다.
   - 출시일: 2004-04-02
   - 평점: 7.21
   ![Maria Full of Grace](https://image.tmdb.org/t/p/w780/30CImATfvHWLXy6a3KmHXnYXB6c.jpg)

3. **A Nightmare on Elm Street** (ID: 377)
   - 개요: 작은 마을의 청소년들이 의식적으로 자살하는 이면에, 아동 성범죄자 프레드 크루거가 나타나 복수를 하며 꿈 속에서 그들의 생명을 위협합니다.
   - 출시일: 1984-11-09
   - 평점: 7.346
   ![A Nightmare on Elm Street](https://image.tmdb.org/t/p/w780/wGTpGGRMZmyFCcrY2YoxVTIBlli.jpg)

4. **Reservoir Dogs** (ID: 500)
   - 개요: 실패한 강도가 경찰의 내막을 제보한 것으로 의심받고, 생존자들이 서로를 의심하며 벌어지는 갈등을 다룬 범죄 드라마입니다.
   - 출시일: 1992-09-02
   - 평점: 8.115
   ![Reservoir 

'영화 **Obsession**와 비슷한 추천 영화 목록은 다음과 같습니다:\n\n1. **Wolf House** (ID: 433854)\n   - 개요: 여섯 명의 친구들이 캠핑 중 사스콧을 발견하고 죽였다고 생각하지만, 실제로 그들이 풀어놓은 것은 더 악랄하고 고대의 초자연적인 공포입니다.\n   - 출시일: 2017-01-03\n   - 평점: 3.4\n   ![Wolf House](https://image.tmdb.org/t/p/w780/cSa5LoVxtHtKOzuOIWcNqZsXkzz.jpg)\n\n2. **Maria Full of Grace** (ID: 436)\n   - 개요: 임신한 콜롬비아 10대가 가족을 위해 절실히 필요한 돈을 벌기 위해 마약 밀매원이 되는 이야기입니다.\n   - 출시일: 2004-04-02\n   - 평점: 7.21\n   ![Maria Full of Grace](https://image.tmdb.org/t/p/w780/30CImATfvHWLXy6a3KmHXnYXB6c.jpg)\n\n3. **A Nightmare on Elm Street** (ID: 377)\n   - 개요: 작은 마을의 청소년들이 의식적으로 자살하는 이면에, 아동 성범죄자 프레드 크루거가 나타나 복수를 하며 꿈 속에서 그들의 생명을 위협합니다.\n   - 출시일: 1984-11-09\n   - 평점: 7.346\n   ![A Nightmare on Elm Street](https://image.tmdb.org/t/p/w780/wGTpGGRMZmyFCcrY2YoxVTIBlli.jpg)\n\n4. **Reservoir Dogs** (ID: 500)\n   - 개요: 실패한 강도가 경찰의 내막을 제보한 것으로 의심받고, 생존자들이 서로를 의심하며 벌어지는 갈등을 다룬 범죄 드라마입니다.\n   - 출시일: 1992-09-02\n   - 평점: 8.115\n   ![Reservoir Dogs](https://image.tmdb.org/t/p/w780/xi8Iu

## 5. (선택) 메모리 확인

`messages` 에 system → user → assistant(tool_calls) → tool → assistant … 가 어떻게 쌓였는지 눈으로.

In [ ]:
for m in messages:
    role = m["role"] if isinstance(m, dict) else m.role
    if role == "tool":
        body = (m["content"] if isinstance(m, dict) else m.content)[:50]
    elif role == "assistant" and not isinstance(m, dict) and m.tool_calls:
        body = "[tool_calls] " + ", ".join(tc.function.name for tc in m.tool_calls)
    else:
        body = (m["content"] if isinstance(m, dict) else m.content) or ""
        body = body[:50]
    print(f"[{role}] {body}")

[system] 너는 영화 전문가 에이전트다.
- 사용자 질문에 답하려면 등록된 도구로 실제 데이터를 가져
[user] 지금 인기 있는 영화 알려줘
[assistant] [tool_calls] get_popular_movies
[tool] [{"adult": false, "backdrop_path": "https://image.
[assistant] 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession** (ID: 1
[user] 그 중 첫 번째 영화 자세히 알려줘
[assistant] [tool_calls] get_movie_details
[tool] {"adult": false, "backdrop_path": "https://image.t
[assistant] 영화 **Obsession** (ID: 1339713)의 자세한 정보는 다음과 같습니다:

[user] 그 영화랑 비슷한 영화 추천해줘
[assistant] [tool_calls] get_similar_movies
[tool] [{"adult": false, "backdrop_path": "https://image.
[assistant] 영화 **Obsession**와 비슷한 추천 영화 목록은 다음과 같습니다:

1. **Wo
